# SOCIOWEAVE — E1 Colab T4 infrastructure preflight

**Purpose:** verify an immutable repository checkout, an NVIDIA T4 runtime, local model loading/inference, and intake of the five operator-supplied frozen benchmark snapshots. This is an **infrastructure and data-intake preflight only**. It does not validate the scientific correctness of every strategy implementation, execute the E1 campaign, or generate scientific results. A passing notebook must not be reported as `E1_READY`.

Run only in a Colab runtime configured with a **T4 GPU**. Benchmark files must be obtained, versioned, split-selected, and license-reviewed separately; this notebook never downloads, selects, synthesizes, or approves benchmark prompts.

In [ ]:
import os, sys, json, subprocess, pathlib, platform, hashlib, importlib.metadata
from datetime import datetime, timezone
REPO_URL = 'https://github.com/msellamiTN/socioweave-studio.git'
CODE_SHA = 'f7a32f4ea2d437559de822e10b54313de8c8dc8d'  # reviewed immutable commit
ROOT = pathlib.Path('/content/socioweave-e1')
REPO_DIR = ROOT / 'repo'
ARTIFACT_DIR = ROOT / 'artifacts'
DATA_DIR = ROOT / 'datasets'
for p in (ROOT, ARTIFACT_DIR, DATA_DIR): p.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version)
print('Platform:', platform.platform())
print('UTC:', datetime.now(timezone.utc).isoformat())


In [ ]:
if not REPO_DIR.exists():
    subprocess.check_call(['git','clone','--no-checkout',REPO_URL,str(REPO_DIR)])
subprocess.check_call(['git','-C',str(REPO_DIR),'fetch','origin',CODE_SHA])
subprocess.check_call(['git','-C',str(REPO_DIR),'checkout','--detach',CODE_SHA])
actual_sha = subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'],text=True).strip()
assert actual_sha == CODE_SHA, (actual_sha, CODE_SHA)
print('Pinned repository SHA:', actual_sha)
subprocess.check_call([sys.executable,'-m','pip','install','-e',str(REPO_DIR / 'engine') + '[local]'])


In [ ]:
import torch
assert torch.cuda.is_available(), 'CUDA GPU is required; select a Colab GPU runtime.'
gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)
gpu_total_gib = gpu_props.total_memory / (1024**3)
print('GPU:', gpu_name)
print('GPU memory GiB:', round(gpu_total_gib, 2))
assert 'T4' in gpu_name, f'Expected an NVIDIA T4 runtime, received: {gpu_name}'
assert gpu_total_gib >= 14.0, f'Unexpected GPU memory capacity: {gpu_total_gib:.2f} GiB'
print('CUDA:', torch.version.cuda, '| PyTorch:', torch.__version__)
print(subprocess.check_output(['nvidia-smi'],text=True))


## 1. Local model smoke test

The default is `Qwen/Qwen2.5-1.5B-Instruct` in 4-bit NF4 mode. This is an infrastructure smoke test only, not a paper-approved model assignment or E1 result. Record the exact model revision and license decision in the campaign preregistration before scientific use.

In [ ]:
from sociodec.providers.huggingface_local import HuggingFaceLocalProvider
from sociodec.providers.base import ChatMessage, GenerationConfig
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
provider = HuggingFaceLocalProvider(model_id=MODEL_ID, load_in_4bit=True, max_input_tokens=4096)
provider.load()
smoke = provider.generate(
    [ChatMessage('system','Return a short plain-text response.'), ChatMessage('user','Reply with exactly: SOCIOWEAVE_T4_OK')],
    GenerationConfig(temperature=0.0, max_new_tokens=24, seed=17),
)
assert smoke.text.strip(), 'Model returned an empty completion.'
print('Provider:', smoke.provider, '| Model:', smoke.model)
print('Smoke output:', repr(smoke.text))
print('Tokens in/out:', smoke.input_tokens, smoke.output_tokens)
print('Peak allocated GiB:', round(torch.cuda.max_memory_allocated()/(1024**3), 3))
print('Peak reserved GiB:', round(torch.cuda.max_memory_reserved()/(1024**3), 3))


## 2. Supply the five frozen dataset files

Place the operator-provided, license-reviewed JSONL files in `/content/socioweave-e1/datasets/` (or mount Drive and point `DATA_DIR` there). Each row must contain `example_id`, `prompt_text`, `split`, `metadata` and dataset-appropriate structured `reference_data` as required by Table 12. Use the exact source version, split and prompt-selection procedure approved for the study; do not assume `test` or select the first N rows. The intake enforces counts (100 each for BREAK, StrategyQA, TaskBench and TravelPlanner; 60 for smart-administration), preserves reference data and hashes provenance. The paper does not fully specify prompt IDs/sampling; document the operator's selection protocol explicitly.

### Optional: stage files from Google Drive

If the five reviewed JSONL files are stored in Drive, mount it and copy them into the notebook's expected `DATA_DIR`. The cell below copies only files that are present and does not overwrite existing local files. It does not download, create, repair, or approve datasets. If you use another location, set `DATA_DIR` explicitly before running the intake cell.

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')
DRIVE_DATA_DIR = pathlib.Path('/content/drive/MyDrive/socioweave-e1/datasets')
EXPECTED_DATASET_NAMES = [
    'BREAK.jsonl', 'StrategyQA.jsonl', 'TaskBench.jsonl',
    'TravelPlanner.jsonl', 'smart-administration.jsonl',
]
if not DRIVE_DATA_DIR.is_dir():
    raise FileNotFoundError(
        f'Drive dataset folder not found: {DRIVE_DATA_DIR}. '
        'Create this folder in Google Drive or edit DRIVE_DATA_DIR.'
    )
staged, already_present, absent_in_drive = [], [], []
for filename in EXPECTED_DATASET_NAMES:
    source = DRIVE_DATA_DIR / filename
    destination = DATA_DIR / filename
    if destination.is_file():
        already_present.append(filename)
    elif source.is_file():
        shutil.copy2(source, destination)
        staged.append(filename)
    else:
        absent_in_drive.append(filename)
print('Copied from Drive:', staged or 'none')
print('Already present locally (not overwritten):', already_present or 'none')
print('Not found in Drive:', absent_in_drive or 'none')
print('Dataset directory:', DATA_DIR)
print('Files currently present:', sorted(p.name for p in DATA_DIR.glob('*.jsonl')))


In [ ]:
from sociodec.datasets import E1DatasetSource, freeze_e1_benchmark_snapshot
DATASET_FILES = {
    'BREAK': DATA_DIR/'BREAK.jsonl',
    'StrategyQA': DATA_DIR/'StrategyQA.jsonl',
    'TaskBench': DATA_DIR/'TaskBench.jsonl',
    'TravelPlanner': DATA_DIR/'TravelPlanner.jsonl',
    'smart-administration': DATA_DIR/'smart-administration.jsonl',
}
# Edit these declarations only after checking the exact upstream/source snapshots.
SOURCE_VERSIONS = {name:'REPLACE_WITH_VERIFIED_SOURCE_VERSION' for name in DATASET_FILES}
SELECTED_SPLITS = {name:'REPLACE_WITH_VERIFIED_SPLIT' for name in DATASET_FILES}
SELECTION_PROTOCOLS = {name:'REPLACE_WITH_EXACT_SELECTION_PROTOCOL' for name in DATASET_FILES}
LICENSE_REVIEW_STATUS = {name:'REPLACE_WITH_APPROVED_STATUS' for name in DATASET_FILES}
missing = [f'{name}: {path}' for name, path in DATASET_FILES.items() if not path.is_file()]
if missing:
    present = [f'{name}: {path}' for name, path in DATASET_FILES.items() if path.is_file()]
    found = '\n'.join(f'  - {p.name} ({p.stat().st_size:,} bytes)' for p in sorted(DATA_DIR.glob('*.jsonl')))
    raise FileNotFoundError(
        'E1 dataset intake stopped: required operator-supplied files are missing.\n'
        f'DATA_DIR: {DATA_DIR}\nMissing:\n  - ' + '\n  - '.join(missing) +
        '\nPresent required files:\n  - ' + ('\n  - '.join(present) if present else 'none') +
        '\nJSONL files found in DATA_DIR:\n' + (found if found else '  - none') +
        '\n\nFix: upload the five reviewed JSONL files to DATA_DIR, or run the optional Google Drive staging cell above.\n'
        'Do not create placeholder or synthetic benchmark rows; the intake requires the real frozen source data.'
    )
if any('REPLACE_WITH_' in x for x in SOURCE_VERSIONS.values()) or any('REPLACE_WITH_' in x for x in SELECTED_SPLITS.values()) or any('REPLACE_WITH_' in x for x in SELECTION_PROTOCOLS.values()):
    raise ValueError('Set and verify source version, split and exact selection protocol for every dataset.')
if any(x != 'approved' for x in LICENSE_REVIEW_STATUS.values()):
    raise ValueError("A documented license review must approve each dataset before freezing.")
sources = {name:E1DatasetSource(
    path=DATASET_FILES[name],
    source_version=SOURCE_VERSIONS[name],
    split=SELECTED_SPLITS[name],
    selection_protocol=SELECTION_PROTOCOLS[name],
    license_review_status=LICENSE_REVIEW_STATUS[name],
) for name in DATASET_FILES}
snapshot = freeze_e1_benchmark_snapshot(sources)
manifest_path = ARTIFACT_DIR/'e1_frozen_snapshot_manifest.json'
manifest_path.write_text(snapshot.manifest_json(), encoding='utf-8')
print('Frozen prompts:', len(snapshot.examples))
print('Snapshot SHA256:', snapshot.manifest['snapshot_sha256'])
print('Manifest:', manifest_path)


In [ ]:
import transformers, accelerate
preflight = {
    'purpose':'infrastructure_and_dataset_intake_preflight_only_no_E1_trials',
    'gate_status': {
        'repository_pin': 'PASS',
        't4_runtime': 'PASS',
        'local_model_smoke': 'PASS',
        'frozen_dataset_intake': 'PASS',
        'scientific_e1_campaign': 'NOT_RUN',
        'e1_ready': False,
    },
    'timestamp_utc':datetime.now(timezone.utc).isoformat(),
    'repository_url':REPO_URL,
    'code_sha':actual_sha,
    'gpu_name':gpu_name,
    'gpu_memory_gib':round(gpu_total_gib,3),
    'cuda_version':torch.version.cuda,
    'torch_version':torch.__version__,
    'transformers_version':transformers.__version__,
    'accelerate_version':accelerate.__version__,
    'model_id':MODEL_ID,
    'model_smoke_output':smoke.text,
    'model_smoke_input_tokens':smoke.input_tokens,
    'model_smoke_output_tokens':smoke.output_tokens,
    'snapshot_sha256':snapshot.manifest['snapshot_sha256'],
    'prompt_count':len(snapshot.examples),
}
preflight_path = ARTIFACT_DIR/'t4_preflight_report.json'
preflight_path.write_text(json.dumps(preflight,ensure_ascii=False,indent=2,sort_keys=True)+'\n',encoding='utf-8')
print(json.dumps(preflight,ensure_ascii=False,indent=2))
print('Saved:', preflight_path)


## Exit criteria and scientific boundary

Preflight passes only if: (1) checkout SHA matches exactly; (2) runtime is an NVIDIA T4 with expected memory; (3) local model loads and returns a non-empty response; (4) all five operator-supplied JSONL snapshots pass fail-closed intake and manifest hashing. Save both JSON artifacts with the run records.

**Not performed here:** E1 strategy/baseline campaign, fixed executor evaluation, V3/V4 assessment, human annotation, statistical analysis, or E1–E9 claims. Do not report this preflight as an experiment. The next gate is a deterministic fake-provider end-to-end runtime test, then an explicitly authorized and cost-capped provider smoke and bounded pilot. Neither is an E1 result.